In [ ]:
# NBVAL_SKIP
from jax import config
#config.update("jax_enable_x64", True)
#config.update('jax_num_cpu_devices', 2)

In [ ]:
#NBVAL_SKIP
import os

# Tell XLA to fake 2 host CPU devices
#os.environ['XLA_FLAGS'] = '--xla_force_host_platform_device_count=3'

# Only make GPU 0 and GPU 1 visible to JAX:
#os.environ['CUDA_VISIBLE_DEVICES'] = '1,2'

#os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]   = "false"

import jax

# Now JAX will list two CpuDevice entries
print(jax.devices())
# → [CpuDevice(id=0), CpuDevice(id=1)]

In [ ]:
# NBVAL_SKIP
import os
#os.environ['SPS_HOME'] = '/mnt/storage/annalena_data/sps_fsps'
#os.environ['SPS_HOME'] = '/home/annalena/sps_fsps'
os.environ['SPS_HOME'] = '/Users/annalena/Documents/GitHub/fsps'
#os.environ['SPS_HOME'] = '/export/home/aschaibl/fsps'

# Load ssp template from FSPS

In [ ]:
# NBVAL_SKIP
from rubix.spectra.ssp.factory import get_ssp_template
ssp_fsps = get_ssp_template("FSPS")

In [ ]:
# NBVAL_SKIP
age_values = ssp_fsps.age
print(age_values.shape)

metallicity_values = ssp_fsps.metallicity
print(metallicity_values.shape)

# Configure pipeline

In [ ]:
# NBVAL_SKIP
from rubix.core.pipeline import RubixPipeline
import os
config = {
    "pipeline":{"name": "calc_gradient",},
    
    "logger": {
        "log_level": "DEBUG",
        "log_file_path": None,
        "format": "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    },
    "data": {
        "name": "IllustrisAPI",
        "args": {
            "api_key": os.environ.get("ILLUSTRIS_API_KEY"),
            "particle_type": ["stars"],
            "simulation": "TNG50-1",
            "snapshot": 99,
            "save_data_path": "data",
        },
        
        "load_galaxy_args": {
        "id": 14,
        "reuse": True,
        },
        
        "subset": {
            "use_subset": True,
            "subset_size": 2,
        },
    },
    "simulation": {
        "name": "IllustrisTNG",
        "args": {
            "path": "data/galaxy-id-14.hdf5",
        },
    
    },
    "output_path": "output",

    "telescope":
        {"name": "TESTGRADIENT",
         "psf": {"name": "gaussian", "size": 5, "sigma": 0.6},
         "lsf": {"sigma": 1.2},
         "noise": {"signal_to_noise": 100,"noise_distribution": "normal"},
         },
    "cosmology":
        {"name": "PLANCK15"},
        
    "galaxy":
        {"dist_z": 0.1,
         "rotation": {"type": "edge-on"},
        },
        
    "ssp": {
        "template": {
            "name": "FSPS"
        },
        "dust": {
                "extinction_model": "Cardelli89",
                "dust_to_gas_ratio": 0.01,
                "dust_to_metals_ratio": 0.4,
                "dust_grain_density": 3.5,
                "Rv": 3.1,
            },
    },        
}

In [ ]:
# NBVAL_SKIP
pipe = RubixPipeline(config)
inputdata = pipe.prepare_data()

# Gradient on the spectrum for each wavelenght

In [ ]:
# NBVAL_SKIP
from rubix.pipeline import linear_pipeline as pipeline

pipeline_instance = RubixPipeline(config)

pipeline_instance._pipeline = pipeline.LinearTransformerPipeline(
    pipeline_instance.pipeline_config, 
    pipeline_instance._get_pipeline_functions()
)
pipeline_instance._pipeline.assemble()
pipeline_instance.func = pipeline_instance._pipeline.compile_expression()

In [ ]:
# pick values
initial_age_index = 95
initial_metallicity_index = 4
age0 = age_values[initial_age_index]
Z0   = metallicity_values[initial_metallicity_index]

In [ ]:
print(f"age0 = {age0}, Z0 = {Z0}")

In [ ]:
# NBVAL_SKIP
import jax.numpy as jnp

inputdata.stars.age = jnp.array([age_values[initial_age_index], age_values[initial_age_index]])
inputdata.stars.metallicity = jnp.array([metallicity_values[initial_metallicity_index], metallicity_values[initial_metallicity_index]])
inputdata.stars.mass = jnp.array([[1.0], [1.0]])
inputdata.stars.velocity = jnp.array([[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]])
inputdata.stars.coords = jnp.array([[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]])

In [ ]:
import dataclasses
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

def spectrum_1d(age, Z, base_data, pipeline_instance):
    # broadcast per-star
    nstar = base_data.stars.age.shape[0]
    stars2 = dataclasses.replace(
        base_data.stars,
        age=jnp.full((nstar,), age),
        metallicity=jnp.full((nstar,), Z),
    )
    data2 = dataclasses.replace(base_data, stars=stars2)

    out = pipeline_instance.func(data2)

    cube = out.stars.datacube                        # shape (…, n_lambda)
    # collapse all non-wavelength axes, keep wavelength last
    spec = cube.reshape((-1, cube.shape[-1])).sum(axis=0)

    return jnp.ravel(spec)   

In [ ]:
spec0 = spectrum_1d(age0, Z0, inputdata, pipeline_instance)

In [ ]:
import matplotlib.pyplot as plt
wave = pipe.telescope.wave_seq

In [ ]:
#jac_mean = jax.jit(jax.jacfwd(spectrum_1d))

In [ ]:
from tensorflow_probability.substrates import jax as tfp
tfd = tfp.distributions
tfb = tfp.bijectors

import tqdm
import optax
import flax.linen as nn
from flax.metrics import tensorboard

In [ ]:
class AffineCoupling(nn.Module):
  @nn.compact
  def __call__(self, x, nunits):
    net = nn.leaky_relu(nn.Dense(128)(x))
    net = nn.leaky_relu(nn.Dense(128)(net))
    shift = nn.Dense(nunits)(net)
    scale = nn.softplus(nn.Dense(nunits)(net))
    return  tfb.Chain([ tfb.Shift(shift), tfb.Scale(scale)])

def make_nvp_fn(n_layers=2, d=2):
  # We alternate between permutations and flow layers
  layers = [ tfb.Permute([1,0])(tfb.RealNVP(d//2,
                                            bijector_fn=AffineCoupling(name='affine%d'%i)))
            for i in range(n_layers) ]

  # We build the actual nvp from these bijectors and a standard Gaussian distribution
  nvp = tfd.TransformedDistribution(
              tfd.MultivariateNormalDiag(loc=jnp.zeros(2), scale_diag=0.05*jnp.ones(2)),
              bijector=tfb.Chain([tfb.Shift([5,0.05])] + layers ))
  # Note that we have here added a shift to the bijector
  return nvp

class NeuralSplineFlowSampler(nn.Module):
  @nn.compact
  def __call__(self,  key, n_samples):
    nvp = make_nvp_fn()
    x = nvp.sample(n_samples, seed=key)
    return x, nvp.log_prob(x)

In [ ]:
model = NeuralSplineFlowSampler()
params = model.init(jax.random.PRNGKey(42), jax.random.PRNGKey(1), 16)


In [ ]:
import pandas as pd
from chainconsumer import ChainConsumer, Chain, Truth

# 1) Draw samples from the untrained bounded flow
theta0, logq0 = model.apply(params, key=jax.random.PRNGKey(1), n_samples=500)
df = pd.DataFrame(theta0, columns=["age", "Z"])

# 2) Optional: pick a fiducial point (for synthetic tests use your known truth)
fid_age = age0                      # example: mid of [0, 20]
fid_Z   = Z0                   # example: inside [4.5e-5, 4.5e-2]

# 3) Build the ChainConsumer plot
c = ChainConsumer()
c.add_chain(Chain(samples=df, name="Initial VI"))
c.add_truth(Truth(location={"age": fid_age, "Z": fid_Z}))

fig = c.plotter.plot(figsize="column")


In [ ]:
def log_prior_gaussian(theta_batch,
                       mu_age=6.0, sigma_age=3.0,
                       mu_Z=1.3e-3, sigma_Z=2e-4):
    """Gaussian prior in physical space."""
    age = theta_batch[:, 0]
    Z   = theta_batch[:, 1]
    lp_age = -0.5 * (((age - mu_age) / sigma_age)**2
                     + jnp.log(2*jnp.pi*sigma_age**2))
    lp_Z   = -0.5 * (((Z - mu_Z) / sigma_Z)**2
                     + jnp.log(2*jnp.pi*sigma_Z**2))
    return lp_age + lp_Z  # shape (batch,)


In [ ]:
import jax, jax.numpy as jnp

def log_likelihood(y, s, mask=None):
    """Full-vector Gaussian log-likelihood."""
    if mask is None:
        mask = jnp.ones_like(y)
    r = y - s
    term = (r**2)
    return jnp.sum(term * mask)

def make_batched_loglik(y, base_data, pipeline_instance, mask=None):
    """Returns a function mapping a batch of theta -> per-sample log-likelihood."""
    def one_theta(theta):
        age, Z = theta[0], theta[1]
        s = spectrum_1d(age, Z, base_data, pipeline_instance)  # -> (n_lambda,)
        return log_likelihood(y, s, mask=mask, )
    return jax.vmap(one_theta)  # (batch,2) -> (batch,)


In [ ]:
def make_elbo_fn(y, base_data, pipeline_instance,
                 mask=None, 
                 mu_age=7.0, sigma_age=2.0,
                 mu_Z=0.001, sigma_Z=1e-3):
    batched_loglik = make_batched_loglik(y, base_data,
                                         pipeline_instance, mask)

    def elbo(params, seed, n_samples=128):
        # Draw θ ~ q_φ(θ)
        theta_batch, log_q = model.apply(params, key=seed, n_samples=n_samples)
        # Compute log p(θ)
        log_p = log_prior_gaussian(theta_batch, mu_age, sigma_age, mu_Z, sigma_Z)
        # Compute log p(y|θ)
        log_lik = batched_loglik(theta_batch)
        # ELBO
        elbo_value = jnp.mean(log_lik + log_p - log_q)
        return -elbo_value   # minimize
    return elbo


In [ ]:
# Random key
seed = jax.random.PRNGKey(0)

# Scheduler and optimizer
total_steps = 20_000
lr = 2e-3
#lr_scheduler = optax.piecewise_constant_schedule(
#    init_value=1e-3,
#    boundaries_and_scales={int(total_steps*0.5): 0.2}
#)
optimizer = optax.adam(lr) #lr_scheduler)
opt_state = optimizer.init(params)

# TensorBoard logs
from flax.metrics import tensorboard
summary_writer = tensorboard.SummaryWriter("logs/elbo")


In [ ]:
eps = 1e-6
sigma_obs = jnp.maximum(jnp.abs(spec0) / 1000.0, eps)
y = spec0
base_data = inputdata

In [ ]:
# Build once, outside update_model
elbo = make_elbo_fn(
    y,                      # observed full flux vector
    base_data,
    pipeline_instance,      
)

In [ ]:

@jax.jit
def update_model(params, opt_state, seed):#, n_samples=128):
    # split RNG: first return is new seed you’ll keep, second is used to sample θ
    seed, key = jax.random.split(seed)

    # loss(params) = -ELBO(params, key, n_samples)
    loss, grads = jax.value_and_grad(elbo)(params, key)#, n_samples)

    # apply Adam step; passing params is safest for transforms that need them
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)

    return params, opt_state, loss, seed


In [ ]:
#%load_ext tensorboard
#%tensorboard --logdir=.

In [ ]:
import tqdm

losses = []

for i in tqdm.tqdm(range(total_steps)):
    # one optimization step (minimizes -ELBO)
    params, opt_state, loss, seed = update_model(params, opt_state, seed)

    losses.append(float(loss))

    # log every 10 steps
    if i % 10 == 0:
        summary_writer.scalar("neg_elbo", float(loss), i)
        #summary_writer.scalar("learning_rate", float(lr_scheduler(i)), i)


In [ ]:
# 1) Sample posterior θ = (age, Z)
seed, sub = jax.random.split(seed)
theta, log_q = model.apply(params, key=sub, n_samples=5000)  # theta.shape == (5000, 2)
age = theta[:, 0]
Z   = theta[:, 1]


In [ ]:
c = ChainConsumer()

# fresh RNG split so we don’t reuse training key
seed, sub = jax.random.split(seed)

# sample θ ~ qϕ(θ)
theta, log_q = model.apply(params, key=sub, n_samples=20_000)  # shape (N, 2)
age = theta[:, 0]
Z   = theta[:, 1]

# ChainConsumer expects a pandas DataFrame
df = pd.DataFrame({"age": age, "Z": Z})

# add the VI chain
c.add_chain(Chain(samples=df, name="VI"))

# optional “truth” dot: use known synthetic truth if you have it; else posterior mean
# truth_age, truth_Z = 8.0, 1.0e-2   # <- set these if you know them
#truth_age, truth_Z = float(age.mean()), float(Z.mean())
truth_age, truth_Z = age0, Z0
c.add_truth(Truth(location={"age": truth_age, "Z": truth_Z}))

fig = c.plotter.plot(figsize="column")

In [ ]:
plt.figure(figsize=(7,3))
plt.plot(np.arange(len(losses)), losses, lw=1)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.grid(True)
plt.tight_layout()
plt.show()